# Production & Resilience Patterns
### Circuit Breaker | Retry+Backoff+Jitter | Bulkhead | Timeout | Cache-Aside | Idempotency

> **One coherent system:** ShopFlow -- 500k-user e-commerce platform.

*Run each cell with **Shift + Enter***

## Setup

In [ ]:
from __future__ import annotations
import time
import threading
import random
import hashlib
import uuid
from dataclasses import dataclass, field
from typing import Any
from collections.abc import Callable

---
## 1 · Circuit Breaker

### Mental Model -- 'The Fuse Box'

```
WHAT   Stop calling a failing service immediately instead of waiting
       for timeout on every request. Probe cautiously to detect recovery.
WHY    Without it: 1 service down -> all callers time out -> thread exhaustion
       -> cascading failure -> entire system down.
HOW    Three states:
       CLOSED   -> normal; count failures
       OPEN     -> fail fast immediately; wait for reset timeout
       HALF-OPEN-> probe with 1 request; success -> CLOSED, fail -> OPEN
WHEN   Any remote call: HTTP APIs, DB queries, cache, message brokers
```

```
Normal:    CLOSED --(failures < threshold)--> CLOSED
Failure:   CLOSED --(failures >= threshold)--> OPEN
Waiting:   OPEN   --(timeout elapsed)-------> HALF-OPEN
Probe OK:  HALF-OPEN --(success)------------> CLOSED
Probe bad: HALF-OPEN --(failure)------------> OPEN
```

**Gotcha 1:** Threshold and timeout must be tuned per service's SLA.
**Gotcha 2:** Circuit must be SHARED across all callers, not per-thread.
**Gotcha 3:** Wrap ONLY the I/O call, not business logic around it.
**Gotcha 4:** Half-open should require multiple successes, not just one.

### Real-World Scenario -- ShopFlow + Stripe Outage

**Incident:** Stripe had a 3-minute partial outage.
Each payment attempt timed out after 30 seconds.
With 2000 checkout req/min, ShopFlow's thread pool exhausted in 90 seconds,
taking the **entire** site down -- not just payments.

**Fix:** Circuit breaker on the Stripe client.
After 3 failures, trip OPEN -- checkout gets a clear error immediately,
thread pool stays healthy, site stays up.

In [ ]:
# BEFORE -- each failure waits for the full 30s timeout
# 2000 req/min x 30s timeout = thread pool exhausted in ~90s

def charge_stripe_BAD(amount: float, token: str) -> str:
    time.sleep(0.05)   # simulates slow call during outage
    raise ConnectionError('Stripe API unreachable')

In [ ]:
# AFTER -- Circuit Breaker

class CircuitBreaker:
    CLOSED    = 'closed'
    OPEN      = 'open'
    HALF_OPEN = 'half-open'

    def __init__(self, threshold: int = 5, timeout: float = 30.0,
                 probe_successes: int = 2) -> None:
        self.threshold, self.reset_timeout = threshold, timeout
        self.probe_successes = probe_successes
        self._state    = self.CLOSED
        self._failures = self._probes_ok = 0
        self._opened_at = 0.0
        self._lock = threading.Lock()

    @property
    def state(self) -> str: return self._state

    def call(self, fn: Callable, *args: Any, **kwargs: Any) -> Any:
        with self._lock:
            if self._state == self.OPEN:
                elapsed = time.monotonic() - self._opened_at
                if elapsed >= self.reset_timeout:
                    self._state, self._probes_ok = self.HALF_OPEN, 0
                    print('  [CB] -> HALF-OPEN (probing...)')
                else:
                    raise RuntimeError(
                        f'Circuit OPEN -- failing fast '
                        f'({self.reset_timeout - elapsed:.1f}s until probe)')
        try:
            result = fn(*args, **kwargs)
        except Exception:
            self._on_failure(); raise
        self._on_success()
        return result

    def _on_failure(self) -> None:
        with self._lock:
            self._failures += 1
            if self._state == self.HALF_OPEN or self._failures >= self.threshold:
                self._state, self._opened_at = self.OPEN, time.monotonic()
                self._failures = 0
                print('  [CB] -> OPEN (tripped)')

    def _on_success(self) -> None:
        with self._lock:
            if self._state == self.HALF_OPEN:
                self._probes_ok += 1
                if self._probes_ok >= self.probe_successes:
                    self._state, self._failures = self.CLOSED, 0
                    print('  [CB] -> CLOSED (recovered!)')
            else:
                self._failures = 0


_fail_stripe = True

def stripe_charge(amount: float, token: str) -> str:
    if _fail_stripe: raise ConnectionError('Stripe unreachable')
    return f'txn_{int(time.time())}'


cb = CircuitBreaker(threshold=3, timeout=60.0, probe_successes=2)

for i in range(6):
    try:
        cb.call(stripe_charge, 99.99, 'tok_abc')
    except RuntimeError as e:
        print(f'  Request {i+1}: FAST-FAIL -- {e}  [state: {cb.state}]')
    except ConnectionError as e:
        print(f'  Request {i+1}: STRIPE ERROR -- {e}  [state: {cb.state}]')

print(f'Circuit state: {cb.state}')
print('Thread pool saved -- requests fail IMMEDIATELY, no 30s waits')

### Where This Is Seen in Real Frameworks

| Tool | Implementation |
|------|---------------|
| **tenacity** | `@retry(stop=stop_after_attempt(5), wait=wait_exponential())` |
| **aiobreaker (Python)** | Async circuit breaker for `asyncio` services |
| **resilience4j (JVM)** | `CircuitBreaker.decorateFunction()` |
| **Nginx / Envoy** | Upstream health checks act as circuit breakers at the network layer |
| **Polly (.NET)** | `Policy.Handle<Exception>().CircuitBreaker(5, TimeSpan.FromSeconds(30))` |

---
## 2 · Retry + Exponential Backoff + Jitter

### Mental Model -- 'The Thundering Herd Problem'

```
WHAT   Retry a failed operation with increasing delays and random jitter
       to avoid synchronized retries overloading the server.
WHY    Without jitter: 1000 clients fail -> all retry at t+1s -> 1000
       simultaneous hits -> server crashes again -> retry at t+2s -> repeat.
       This is the 'thundering herd' problem.
HOW    Full jitter (AWS recommended):
       wait = random(0, min(cap, base x 2^attempt))
       The jitter term desynchronizes retries across clients.
WHEN   Any transient-failure-prone I/O: DB connections, HTTP APIs, message brokers.
```

```
Attempt 1: wait 0s   (immediate)
Attempt 2: wait ~1s  + jitter
Attempt 3: wait ~2s  + jitter
Attempt 4: wait ~4s  + jitter
Attempt 5: wait ~8s  + jitter  (capped at max_wait)
```

In [ ]:
# BEFORE -- naive retry (no backoff, no jitter = thundering herd)

def fetch_product_BAD(product_id: int, retries: int = 3) -> dict:
    for attempt in range(retries):
        try:
            return {'id': product_id}
        except Exception:
            time.sleep(1)   # ALL clients retry at exactly t+1s, t+2s, t+3s
                            # 1000 clients -> 1000 simultaneous retries -> crash
    raise RuntimeError('Failed after retries')

In [ ]:
# AFTER -- Exponential Backoff with Full Jitter

def retry_with_backoff(
    fn: Callable,
    *args: Any,
    max_attempts: int = 5,
    base_delay: float = 1.0,
    max_delay: float = 30.0,
    retryable: tuple = (ConnectionError, TimeoutError, OSError),
    **kwargs: Any,
) -> Any:
    # Full jitter: wait = random(0, min(max_delay, base x 2^attempt))
    # Desynchronizes retries from different clients.
    for attempt in range(max_attempts):
        try:
            return fn(*args, **kwargs)
        except retryable as exc:
            if attempt == max_attempts - 1: raise
            cap  = min(max_delay, base_delay * (2 ** attempt))
            wait = random.uniform(0, cap)
            print(f'  Attempt {attempt+1} failed ({exc}). Retrying in {wait:.2f}s...')
            time.sleep(wait)   # production: asyncio.sleep(wait)


# Simulate flaky service that succeeds on attempt 3
_attempt_count = 0
def flaky_api_call(endpoint: str) -> dict:
    global _attempt_count
    _attempt_count += 1
    if _attempt_count < 3:
        raise ConnectionError(f'Service unavailable (attempt {_attempt_count})')
    _attempt_count = 0
    return {'data': endpoint, 'status': 'ok'}


result = retry_with_backoff(flaky_api_call, '/products/1')
print(f'  Success: {result}')

print('\nJitter spread (5 clients, attempt 2):')
for client in range(5):
    wait = random.uniform(0, min(30.0, 1.0 * (2**2)))
    print(f'  Client {client+1}: {wait:.2f}s')
print('  -> Spread across the window; no thundering herd!')

### Where This Is Seen in Real Frameworks

| Library | Usage |
|---------|-------|
| **tenacity** | `@retry(wait=wait_exponential(multiplier=1, max=30), reraise=True)` |
| **boto3** | Built-in retry with adaptive backoff for all AWS API calls |
| **httpx / requests-retry** | `HTTPAdapter(max_retries=Retry(backoff_factor=2))` |
| **Celery** | `@app.task(autoretry_for=(Exception,), retry_backoff=True, max_retries=5)` |
| **aiohttp** | `aiohttp_retry.RetryClient` with exponential strategy |

---
## 3 · Bulkhead

### Mental Model -- 'The Ship's Compartments'

```
WHAT   Isolate resource pools so one failing consumer can't exhaust
       resources needed by other consumers.
WHY    Without bulkheads: a slow /reports endpoint holds 50 threads.
       /checkout has no threads -> site down even though DB is healthy.
HOW    Separate semaphores per endpoint/service.
       /reports gets 5 threads max; /checkout gets 40 threads max.
WHEN   Mixed traffic priorities: user-facing vs batch,
       critical vs non-critical endpoints.
```

```
Without bulkhead:            With bulkhead:
  All 50 threads             /checkout: 40 threads (protected)
      |                      /reports:   5 threads (isolated)
      +- /checkout x5             |
      +- /reports x45 (slow!)     -> /checkout unaffected
      -> /checkout STARVED!
```

In [ ]:
class Bulkhead:
    # Limits concurrent calls using a semaphore

    def __init__(self, name: str, max_concurrent: int) -> None:
        self.name    = name
        self._sem    = threading.Semaphore(max_concurrent)
        self._max    = max_concurrent
        self._active = 0
        self._lock   = threading.Lock()

    def __enter__(self):
        acquired = self._sem.acquire(blocking=False)
        if not acquired:
            raise RuntimeError(f"Bulkhead '{self.name}' full "
                               f'({self._max} concurrent calls in flight)')
        with self._lock: self._active += 1
        return self

    def __exit__(self, *_):
        with self._lock: self._active -= 1
        self._sem.release()


checkout_bulkhead = Bulkhead('checkout', max_concurrent=40)
reports_bulkhead  = Bulkhead('reports',  max_concurrent=5)


def generate_report(report_id: str) -> str:
    with reports_bulkhead:
        time.sleep(0.01)  # slow batch operation
        return f'Report {report_id} generated'


results: list = []
errors:  list = []

def run_report(i):
    try: results.append(generate_report(f'R{i}'))
    except RuntimeError as e: errors.append(str(e))

threads = [threading.Thread(target=run_report, args=(i,)) for i in range(10)]
for t in threads: t.start()
for t in threads: t.join()

print(f'Reports succeeded: {len(results)}')
print(f'Reports rejected (bulkhead full): {len(errors)}')
print(f'Checkout bulkhead still free: {checkout_bulkhead._sem._value}/40 slots')
print('/checkout is completely unaffected by /reports overload')

### Where This Is Seen in Real Frameworks

| Tool | Usage |
|------|-------|
| **asyncio.Semaphore** | `async with asyncio.Semaphore(10): await call()` |
| **Gunicorn workers** | `--workers=4` + `--threads=2` -- separate pools per worker |
| **ThreadPoolExecutor** | `max_workers=10` per pool type |
| **Nginx** | `limit_conn_zone` -- per-endpoint connection limits |
| **Envoy / Istio** | Circuit breaking + bulkhead via connection pool settings |

---
## 4 · Cache-Aside (Lazy Loading)

### Mental Model -- 'The Cheat Sheet'

```
WHAT   The application manages the cache explicitly:
       check cache -> miss -> load from DB -> populate cache -> return.
WHY    Not all data is worth caching; cache-aside puts the app in control.
HOW    Read: check cache, miss -> DB -> write to cache.
       Write: invalidate cache entry.
WHEN   Product catalogs | user profiles | config | reference data.
       Avoid for data that changes faster than the TTL.
```

```
READ:
  App -> cache.get(key)
           | HIT  -> return cached value
           | MISS -> DB.query() -> cache.set(key, val, ttl) -> return

WRITE:
  App -> DB.update(entity)
       -> cache.delete(key)   <- invalidate; next read re-populates
```

**Gotcha -- Cache Stampede:** Many requests miss simultaneously (cold start
or TTL expiry), all hitting DB at once.
Fix: probabilistic early expiry (XFetch) or distributed lock.

In [ ]:
class CacheAside:
    def __init__(self, ttl: float = 60.0) -> None:
        self._store: dict[str, tuple[Any, float]] = {}
        self._ttl   = ttl
        self._lock  = threading.Lock()

    def get(self, key: str) -> Any:
        with self._lock:
            if key in self._store:
                val, expires_at = self._store[key]
                if time.monotonic() < expires_at: return val
                del self._store[key]
        return None

    def set(self, key: str, val: Any) -> None:
        with self._lock:
            self._store[key] = (val, time.monotonic() + self._ttl)

    def delete(self, key: str) -> None:
        with self._lock: self._store.pop(key, None)


_cache = CacheAside(ttl=60.0)

def get_product(product_id: int) -> dict:
    key = f'product:{product_id}'
    cached = _cache.get(key)
    if cached is not None:
        print(f'  [Cache] HIT  product:{product_id}'); return cached
    print(f'  [DB]    QUERY product:{product_id}')
    product = {'id': product_id, 'name': 'Widget', 'price': 9.99}
    _cache.set(key, product)
    return product

def update_product(product_id: int, updates: dict) -> dict:
    key = f'product:{product_id}'
    print(f'  [DB]    UPDATE product:{product_id}')
    _cache.delete(key)  # invalidate on write
    print(f'  [Cache] INVALIDATED product:{product_id}')
    return {'id': product_id, **updates}


print('First read (miss):')  ; get_product(1)
print('Second read (hit):')  ; get_product(1)
print('\nUpdate product:')    ; update_product(1, {'name': 'Widget Pro', 'price': 14.99})
print('\nRead after update:') ; print(' ', get_product(1))

### Where This Is Seen in Real Frameworks

| Tool | Usage |
|------|-------|
| **Django cache framework** | `cache.get(key)` / `cache.set(key, val, timeout)` |
| **Flask-Caching** | `@cache.cached(timeout=300, key_prefix='product_%s')` |
| **Redis-py** | `r.get(key)` -> miss -> query -> `r.setex(key, ttl, val)` |
| **SQLAlchemy** | Second-level cache via `dogpile.cache` |
| **FastAPI** | `functools.lru_cache` on startup-loaded data; Redis for distributed cache |

---
## 5 · Idempotency Key

### Mental Model -- 'The Postal Tracking Number'

```
WHAT   Assign a unique key to each operation so retries are safe.
       Repeating the same request produces the same result.
WHY    Retries are inevitable (network timeouts, user double-clicks).
       Without idempotency: double charges, duplicate orders, ghost emails.
HOW    Client generates a UUID per request. Server stores result under that key.
       On retry: key found -> return stored result (no re-execution).
WHEN   Payments | order placement | email sends | any non-idempotent mutation.
```

```
Client --POST /checkout (Idempotency-Key: uuid-1234)--> Server
         timeout! retry...
Client --POST /checkout (Idempotency-Key: uuid-1234)--> Server
           | key found in store -> return SAME response  (no double charge!)
```

In [ ]:
@dataclass
class IdempotencyStore:
    # In production: Redis with TTL. Here: in-memory dict.
    _store: dict[str, dict] = field(default_factory=dict)

    def get(self, key: str) -> dict | None:
        return self._store.get(key)

    def save(self, key: str, result: dict) -> None:
        self._store[key] = result   # production: SETEX with 24h TTL


_idempotency = IdempotencyStore()


def place_order(order_data: dict, idempotency_key: str) -> dict:
    existing = _idempotency.get(idempotency_key)
    if existing:
        print(f'  [Idempotent] Key {idempotency_key[:8]}... already processed -- '
              f'returning cached result (no double-charge!)')
        return existing

    order_id  = f'ord_{hashlib.sha256(idempotency_key.encode()).hexdigest()[:8]}'
    charge_id = f'txn_{int(time.time())}'
    result = {'order_id': order_id, 'charge_id': charge_id,
              'status': 'created', 'idempotency_key': idempotency_key}
    _idempotency.save(idempotency_key, result)
    print(f'  [New]       Created order {order_id}, charged {charge_id}')
    return result


idem_key = str(uuid.uuid4())
cart = {'items': [{'sku': 'W1', 'qty': 2}], 'total': 19.99}

print('First request (new):')
r1 = place_order(cart, idem_key)

print('\nRetry after timeout (same key):')
r2 = place_order(cart, idem_key)

print('\nSame result returned?', r1['order_id'] == r2['order_id'])
print('Charge IDs identical?',   r1['charge_id'] == r2['charge_id'])
print('No double charge!')

### Where This Is Seen in Real Frameworks

| Tool | Usage |
|------|-------|
| **Stripe API** | `Idempotency-Key` header -- Stripe stores results for 24 hours |
| **PayPal** | `PayPal-Request-Id` header -- same mechanism |
| **AWS SQS** | Message deduplication ID -- exactly-once delivery within dedup window |
| **Celery** | Task IDs -- same task ID won't re-run if result is cached |
| **Kafka** | Idempotent producer -- `enable.idempotence=true` |
| **Django** | Custom `IdempotencyKey` model + middleware |